# 06 - Actor Profiles, Networks, and Typology

                This notebook builds person-topic profiles and network metrics for recurring contributors. The typology is heuristic and should only be interpreted for manually checked recurring names.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from hsr_analysis.common import *

import numpy as np
import pandas as pd

ensure_project_structure(ROOT)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

print(f"Project root: {ROOT}")

In [ ]:
from itertools import combinations

import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns

articles = pd.read_csv(ROOT / "outputs/tables/articles_clean.csv")
article_topics = pd.read_csv(ROOT / "outputs/tables/article_topics.csv")
article_persons = pd.read_csv(ROOT / "outputs/tables/article_persons.csv")
persons = pd.read_csv(ROOT / "outputs/tables/persons_clean.csv")
embedding_index = pd.read_csv(ROOT / "outputs/tables/article_embedding_index.csv")
embeddings = np.load(ROOT / "outputs/models/article_embeddings_main.npy")

person_articles = article_persons.merge(
    article_topics[["article_id", "topic_id", "topic_label_human", "topic_macro_category"]],
    on="article_id",
    how="left",
).merge(articles[["article_id", "year", "issue_id"]], on="article_id", how="left", suffixes=("", "_article"))
embedding_lookup = dict(zip(embedding_index["article_id"], embedding_index["embedding_row"]))

## Person-Topic Profiles

In [ ]:
profile_rows = []
person_topic_counts = (
    person_articles.groupby(["person_id", "person_name", "topic_id", "topic_label_human"])
    .size()
    .reset_index(name="n_articles_in_topic")
)
person_totals = person_articles.groupby("person_id")["article_id"].nunique().rename("n_articles_total").reset_index()
person_topic_profiles_long = person_topic_counts.merge(person_totals, on="person_id", how="left")
person_topic_profiles_long["topic_share_person"] = person_topic_profiles_long["n_articles_in_topic"] / person_topic_profiles_long["n_articles_total"]
write_csv(person_topic_profiles_long, ROOT / "outputs/tables/person_topic_matrix_long.csv")

for person_id, group in person_articles.groupby("person_id"):
    group = group.drop_duplicates("article_id")
    topic_counts = group["topic_id"].value_counts()
    dominant_topic = group["topic_label_human"].value_counts().index[0] if group["topic_label_human"].notna().any() else pd.NA
    dominant_share = float(topic_counts.iloc[0] / topic_counts.sum()) if len(topic_counts) else np.nan
    entropy_norm = normalized_entropy(group["topic_id"], n_categories=article_topics["topic_id"].nunique())
    rows = [embedding_lookup.get(aid) for aid in group["article_id"] if aid in embedding_lookup]
    dispersion = cosine_centroid_dispersion(embeddings[rows]) if len(rows) else np.nan
    macro = group["topic_macro_category"].fillna("")
    profile_rows.append(
        {
            "person_id": person_id,
            "person_name": group["person_name"].iloc[0],
            "n_articles": group["article_id"].nunique(),
            "first_year": group["year"].min(),
            "last_year": group["year"].max(),
            "active_span": group["year"].max() - group["year"].min() + 1,
            "n_topics": group["topic_id"].nunique(),
            "dominant_topic": dominant_topic,
            "dominant_topic_share": dominant_share,
            "topic_entropy_norm": entropy_norm,
            "effective_n_topics": float(np.exp(shannon_entropy(group["topic_id"]))),
            "method_share": float(macro.eq("methods_and_methodology").mean()),
            "data_infrastructure_share": float(macro.eq("data_and_infrastructure").mean()),
            "substantive_topic_share": float((~macro.isin(["methods_and_methodology", "data_and_infrastructure", ""])).mean()),
            "semantic_dispersion_highdim": dispersion,
        }
    )
profiles = pd.DataFrame(profile_rows).merge(
    persons[["person_id", "is_editor_any", "is_special_issue_editor", "is_managing_editor", "editor_role_certainty"]],
    on="person_id",
    how="left",
)
profiles[["is_editor_any", "is_special_issue_editor", "is_managing_editor"]] = profiles[["is_editor_any", "is_special_issue_editor", "is_managing_editor"]].fillna(False)
profiles["editor_role_certainty"] = profiles["editor_role_certainty"].fillna("unknown")
write_csv(profiles, ROOT / "outputs/tables/person_topic_profiles.csv")
display(profiles.sort_values("n_articles", ascending=False).head(20))

## Networks

In [ ]:
G = nx.Graph()
for article_id, group in person_articles.groupby("article_id"):
    people = sorted(set(group["person_id"].dropna()))
    for person in people:
        G.add_node(person)
    for a, b in combinations(people, 2):
        if G.has_edge(a, b):
            G[a][b]["weight"] += 1
        else:
            G.add_edge(a, b, weight=1)

if G.number_of_nodes():
    degree = dict(G.degree())
    weighted_degree = dict(G.degree(weight="weight"))
    betweenness = nx.betweenness_centrality(G, weight="weight", normalized=True) if G.number_of_edges() else {n: 0 for n in G.nodes}
    pagerank = nx.pagerank(G, weight="weight") if G.number_of_edges() else {n: 1 / G.number_of_nodes() for n in G.nodes}
    components = {node: cid for cid, comp in enumerate(nx.connected_components(G)) for node in comp}
else:
    degree = weighted_degree = betweenness = pagerank = components = {}

participation = {}
for person_id, group in person_articles.groupby("person_id"):
    counts = group["topic_id"].value_counts()
    total = counts.sum()
    participation[person_id] = float(1 - ((counts / total) ** 2).sum()) if total else 0.0

network_metrics = pd.DataFrame(
    [
        {
            "person_id": node,
            "degree_centrality": degree.get(node, 0),
            "weighted_degree": weighted_degree.get(node, 0),
            "betweenness_centrality": betweenness.get(node, 0),
            "pagerank": pagerank.get(node, 0),
            "participation_coefficient_across_topics": participation.get(node, 0),
            "brokerage_score": betweenness.get(node, 0) * participation.get(node, 0),
            "network_component": components.get(node, -1),
        }
        for node in set(list(degree.keys()) + list(profiles["person_id"]))
    ]
).merge(profiles[["person_id", "person_name", "n_articles"]], on="person_id", how="left")
write_csv(network_metrics, ROOT / "outputs/tables/person_network_metrics.csv")

fig, ax = plt.subplots(figsize=(9, 7))
core_nodes = [n for n, d in degree.items() if d >= 2]
sub = G.subgraph(core_nodes).copy()
if sub.number_of_nodes():
    pos = nx.spring_layout(sub, seed=42, k=0.35)
    article_count_lookup = profiles.drop_duplicates("person_id").set_index("person_id")["n_articles"].to_dict()
    sizes = [max(20, article_count_lookup.get(n, 1) * 18) for n in sub.nodes]
    nx.draw_networkx_edges(sub, pos, ax=ax, alpha=0.18, width=0.8)
    nx.draw_networkx_nodes(sub, pos, ax=ax, node_size=sizes, node_color="#4c78a8", alpha=0.75)
ax.set_title("Co-authorship network core")
ax.axis("off")
fig.tight_layout()
fig.savefig(ROOT / "outputs/figures/fig_11_coauthorship_network_core.png", dpi=220)
save_caption(ROOT, "fig_11_coauthorship_network_core.png", "Core co-authorship network; network measures are sensitive to name disambiguation and corpus coverage.")
plt.show()
display(network_metrics.sort_values("brokerage_score", ascending=False).head(20))

## Actor Typology

In [ ]:
typed = profiles.merge(
    network_metrics[["person_id", "betweenness_centrality", "participation_coefficient_across_topics", "brokerage_score"]],
    on="person_id",
    how="left",
)
typed[["betweenness_centrality", "participation_coefficient_across_topics", "brokerage_score"]] = typed[["betweenness_centrality", "participation_coefficient_across_topics", "brokerage_score"]].fillna(0)

entropy_q67 = typed["topic_entropy_norm"].quantile(0.67)
broker_q67 = typed["brokerage_score"].quantile(0.67)
disp_q67 = typed["semantic_dispersion_highdim"].quantile(0.67)

def assign_type(row):
    if row["n_articles"] < 2:
        return "One-time contributors"
    if bool(row.get("is_editor_any", False)) and (row["active_span"] >= 10 or row["n_articles"] >= 3):
        return "Institutional anchors"
    if row["n_articles"] >= 2 and row["method_share"] + row["data_infrastructure_share"] >= 0.5:
        return "Methods and data architects"
    if row["n_articles"] >= 3 and row["dominant_topic_share"] >= 0.60 and row["topic_entropy_norm"] <= 0.35:
        return "Thematic specialists"
    if row["n_articles"] >= 3 and row["topic_entropy_norm"] >= entropy_q67 and row["brokerage_score"] >= broker_q67 and row["semantic_dispersion_highdim"] <= disp_q67:
        return "Bridge builders"
    if row["n_articles"] >= 3 and row["topic_entropy_norm"] >= entropy_q67 and row["semantic_dispersion_highdim"] >= disp_q67:
        return "Wide-ranging contributors"
    return "Recurring contributors"

typed["actor_type"] = typed.apply(assign_type, axis=1)
typed["typology_uncertainty"] = np.where(typed["n_articles"] < 3, "high: fewer than three articles", "moderate")
typed["topic_entrepreneur_candidate"] = False
write_csv(typed, ROOT / "outputs/tables/person_typology.csv")

sensitivity = typed[["person_id", "person_name", "actor_type", "n_articles", "topic_entropy_norm", "brokerage_score", "semantic_dispersion_highdim", "typology_uncertainty"]].copy()
write_csv(sensitivity, ROOT / "outputs/tables/person_typology_sensitivity.csv")

plot_typed = typed[typed["n_articles"] >= 2].copy()
fig, ax = plt.subplots(figsize=(10, 7))
sns.scatterplot(
    data=plot_typed,
    x="topic_entropy_norm",
    y="brokerage_score",
    hue="actor_type",
    style="is_editor_any",
    size="n_articles",
    sizes=(30, 350),
    alpha=0.8,
    ax=ax,
)
ax.set_title("Actor typology matrix")
ax.set_xlabel("Thematic breadth")
ax.set_ylabel("Brokerage score")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
fig.tight_layout()
fig.savefig(ROOT / "outputs/figures/fig_13_actor_typology_matrix.png", dpi=220)
save_caption(ROOT, "fig_13_actor_typology_matrix.png", "Recurring contributors positioned by thematic breadth and brokerage; types are heuristic and uncertainty depends on name and role validation.")
plt.show()
display(typed["actor_type"].value_counts())